# Level Zero baseline: Muon + Auxiliary AdamW

Runs the three matched seeds for **`muon`** on the common 4-layer, 4-head, width-128 nanoGPT protocol. Trajectory shading is a **Bollinger-style across-seed envelope** (mean ± 2 sample SD), not a rolling smoother. Final test error bars are 95% Student-t intervals across three seeds. The test split is used only for final and validation-selected checkpoints.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
if (cwd / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd
elif (cwd.parent / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd.parent
elif (cwd / "level_0_baseline" / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd / "level_0_baseline"
else:
    raise FileNotFoundError("Run from the repository or level_0_baseline tree")

sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from level0_baseline.analysis import (
    load_metrics, load_spectral_metrics, load_test_results,
    plot_final_test_ci, plot_optimizer_band, run_directory,
    run_status_table, test_summary_table,
)
from level0_baseline.config import canonical_seeds, load_config
from level0_baseline.generate import generate_from_checkpoint, write_samples
from level0_baseline.runner import run_suite

CONFIG_PATH = EXPERIMENT_ROOT / "configs" / "level0.yaml"
CONFIG = load_config(CONFIG_PATH)
SEEDS = canonical_seeds(CONFIG)
ROOT = Path(os.environ.get("NANOGPT_LEVEL0_ROOT", "/tmp/nanogpt-level0-baselines"))
DATA_ROOT = Path(os.environ.get("NANOGPT_LEVEL0_DATA_ROOT", ROOT / "data"))
RESULTS_ROOT = Path(os.environ.get("NANOGPT_LEVEL0_RESULTS_ROOT", ROOT / "results"))
DEVICE = os.environ.get("NANOGPT_LEVEL0_DEVICE", "mps")
BAND_SIGMA = float(CONFIG["analysis"]["bollinger_sigma"])
print(EXPERIMENT_ROOT, DATA_ROOT, RESULTS_ROOT, DEVICE, sep="\n")

In [ ]:
OPTIMIZER = "muon"
run_status_table(RESULTS_ROOT, optimizers=[OPTIMIZER], seeds=SEEDS)

## Run or resume the three seeds

Runs are sequential to avoid simultaneous MPS pressure. Completed runs are skipped; incomplete runs resume from `checkpoint_latest.pt`.

In [ ]:
RUN_TRAINING = True
if RUN_TRAINING:
    display(run_suite(
        config_path=CONFIG_PATH, data_root=DATA_ROOT, results_root=RESULTS_ROOT,
        optimizers=[OPTIMIZER], seeds=SEEDS, device=DEVICE,
        resume=True, generate=False,
    ))

In [ ]:
metrics = load_metrics(RESULTS_ROOT, optimizers=[OPTIMIZER], seeds=SEEDS)
spectral = load_spectral_metrics(RESULTS_ROOT, optimizers=[OPTIMIZER], seeds=SEEDS)
test_results = load_test_results(RESULTS_ROOT, optimizers=[OPTIMIZER], seeds=SEEDS)
test_summary = test_summary_table(test_results)
display(metrics.groupby("seed").tail(1)[[
    "seed", "step", "epoch", "train_loss", "train_accuracy",
    "val_loss", "val_perplexity", "val_accuracy", "test_loss",
    "test_perplexity", "test_accuracy"
]])

## Learning and optimization trajectories

In [ ]:
for metric in [
    "train_loss", "train_perplexity", "train_accuracy",
    "val_loss", "val_perplexity", "val_accuracy",
    "val_generalization_gap", "grad_norm_pre_clip",
    "update_to_weight_ratio", "tokens_per_sec",
]:
    plot_optimizer_band(
        metrics, optimizer=OPTIMIZER, metric=metric, sigma=BAND_SIGMA,
        title=f"{OPTIMIZER}: {metric} (mean ± 2 SD)",
    )
    plt.show()

## WeightWatcher diagnostics

Only values returned by `watcher.analyze(ERG=True)` are plotted. Missing alpha or ERG-gap values remain missing; no fallback is synthesized.

In [ ]:
for metric in ["alpha_median", "ERG_gap_median", "D_median", "stable_rank_median"]:
    plot_optimizer_band(
        spectral, optimizer=OPTIMIZER, metric=metric, sigma=BAND_SIGMA,
        title=f"{OPTIMIZER}: {metric} (mean ± 2 SD)",
    )
    if metric == "alpha_median":
        plt.axhline(2.0, linestyle="--", linewidth=1.0)
    if metric == "ERG_gap_median":
        plt.axhline(0.0, linestyle="--", linewidth=1.0)
    plt.show()

## Held-out test metrics

In [ ]:
display(test_summary[[
    "optimizer_label", "checkpoint", "metric", "n", "mean", "sd",
    "ci95_half_width", "ci95_lower", "ci95_upper"
]])
for metric in ["test_loss", "test_perplexity", "test_accuracy"]:
    plot_final_test_ci(test_summary, metric=metric, checkpoint="final", optimizers=[OPTIMIZER])
    plt.show()

## Generate text from each final checkpoint

This qualitative diagnostic complements, but does not replace, held-out loss, perplexity, and accuracy.

In [ ]:
RUN_GENERATION = True
sampling = CONFIG["sampling"]
if RUN_GENERATION:
    for seed in SEEDS:
        run_dir = run_directory(RESULTS_ROOT, OPTIMIZER, seed)
        checkpoint = run_dir / "checkpoint_final.pt"
        sample_seed = int(sampling["seed_offset"]) + int(seed)
        samples = generate_from_checkpoint(
            checkpoint, prompt=str(sampling["prompt"]),
            num_samples=int(sampling["num_samples"]),
            max_new_tokens=int(sampling["max_new_tokens"]),
            temperature=float(sampling["temperature"]), top_k=int(sampling["top_k"]),
            seed=sample_seed, device=DEVICE,
        )
        write_samples(
            run_dir, samples, prompt=str(sampling["prompt"]), checkpoint=checkpoint,
            settings={**sampling, "device": DEVICE, "seed": sample_seed},
        )
        display(Markdown(f"### Seed {seed}"))
        for index, sample in enumerate(samples, 1):
            display(Markdown(f"**Sample {index}**\n\n{sample}"))